# 25. Classify Structured Data Using Keras Preprocessing Layers

This tutorial demonstrates how to classify structured data, such as tabular data, using a simplified version of the PetFinder dataset from a Kaggle competition stored in a CSV file.

You will use Keras to define the model, and Keras preprocessing layers as a bridge to map from columns in a CSV file to features used to train the model. The goal is to predict if a pet will be adopted.

This tutorial contains complete code for:

* Loading a CSV file into a DataFrame using pandas.
* Building an input pipeline to batch and shuffle the rows using `tf.data`.
* Mapping from columns in the CSV file to features used to train the model with the Keras preprocessing layers.
* Building, training, and evaluating a model using Keras built-in methods.

**Note:** This tutorial is similar to Classift structured data with feature columns. This version uses the Keras preprocessing layers instead of the `tf.feature_column` API, as the former are more intuitive and can be easily included inside your model to simplify deployment.

### The PetFinder.my mini dataset

There are several thousand rows in the PetFinder.my mini's CSV dataset file, where each row describes a pet (a dog or a cat) and each column describes an attribute, such as age, breed, color, and so on.

In the dataset's summary below, notice there are mostly numerical and categorical columns. In this tutorial, you will only be dealing with those two feature types, dropping `Description` (a free text feature) and `AdoptionSpeed` (a classification feature) during data preprocessing.

| Column | Pet Description | Feature Type | Data Type |
|--------|-----------------|--------------|-----------|
| `Type` | Type of animal (Dog, Cat) | Categorical | String |
| `Age` | Age | Numerical | Integer |
| `Breed1` | Primary breed | Categorical | String |
| `Color1` | Color 1 | Categorical | String |
| `Color2` | Color 2 | Categorical | String |
| `MaturitySize` | Size at maturity | Categorical | String |
| `FurLength` | Fur length | Categorical | String |
| `Vaccinated` | Pet has been vaccinated | Categorical | String |
| `Sterilized` | Pet has been sterilized | Categorical | String |
| `Health` | Health condition | Categorical | String |
| `Fee` | Adoption fee | Numerical | Integer |
| `Description` | Profile write-up | Text | String |
| `PhotoAmt` | Total uploaded photos | Numerical | Integer |
| `AdoptionSpeed` | Categorical speed of adoption | Classification | Integer |

### Import TensorFlow and other libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers

In [2]:
tf.__version__

'2.20.0'

### Load the dataset and read it into a pandas DataFrame

pandas is a Python library with many helpful utilities for loading and working with structured data. Use `tf.keras.utils.get_file` to download and extract the CSV file with the PetFinder.my mini dataset, and load it into a DataFrame with `pandas.read_csv`:

In [3]:
dataset_url = 'http://storage.googleapis.com/download.tensorflow.org/data/petfinder-mini.zip'
csv_file = 'datasets/petfinder_mini_extracted/petfinder-mini/petfinder-mini.csv'

tf.keras.utils.get_file('petfinder_mini.zip', dataset_url, extract=True, cache_dir='.')

'.\\datasets\\petfinder_mini_extracted'

In [4]:
dataframe = pd.read_csv(csv_file)

Inspect the dataset by checking the first five rows of the DataFrame:

In [5]:
dataframe.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,Description,PhotoAmt,AdoptionSpeed
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,Nibble is a 3+ month old ball of cuteness. He ...,1,2
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,I just found it alone yesterday near my apartm...,2,0
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,Their pregnant mother was dumped by her irresp...,7,3
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,"Good guard dog, very alert, active, obedience ...",8,2
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,This handsome yet cute boy is up for adoption....,3,2


In [6]:
dataframe.shape

(11537, 15)

In [7]:
print(int(len(dataframe) * 0.8), int(len(dataframe) * 0.9))

9229 10383


### Create a target variable

The original task in Kaggle's PetFinder.my Adoption Prediction competition was to predict the speed at which a pet will be adopted (e.g., in the first week, the first month, the first three months, and so on).

In this tutorial, you will simplify the task by transforming it into a binary classification problem, where you simply have to predict whether a pet was adopted or not.

After modifying the `AdoptionSpeed` column, `0` will indicate the pet was not adopted, and `1` will indicate it was.

In [8]:
# In the original dataset, 'AdoptionSpeed' of 4 indicates a pet was not adopted.
dataframe['target'] = np.where(dataframe['AdoptionSpeed'] == 4, 0, 1)

# Drop unused features
dataframe = dataframe.drop(columns=['AdoptionSpeed', 'Description'])

### Split the DataFrame into training, validation and test sets

The dataset is in a single pandas DataFrame. Split it into training, validation and test sets using a, for example, 80:10:10 ratio, respectively:

In [9]:
# train, val, test = np.split(dataframe.sample(frac=1), [int(0.8*len(dataframe)), int(0.9*len(dataframe))])
train = dataframe.iloc[0:9229]
val = dataframe.iloc[9229:10383]
test = dataframe.iloc[10383:]

In [10]:
print(len(train), "training examples")
print(len(val), "validation examples")
print(len(test), "test examples")

9229 training examples
1154 validation examples
1154 test examples


In [11]:
train.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,PhotoAmt,target
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,1,1
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,2,1
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,7,1
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,8,1
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,3,1


In [12]:
val.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,PhotoAmt,target
9229,Dog,4,Labrador Retriever,Female,Black,No Color,Large,Short,Yes,No,Healthy,0,3,1
9230,Dog,36,Mixed Breed,Female,Black,White,Medium,Medium,Yes,Yes,Healthy,0,5,1
9231,Dog,12,Terrier,Female,Brown,Cream,Medium,Medium,Not Sure,No,Healthy,0,5,1
9232,Dog,3,Belgian Shepherd Malinois,Male,Black,Brown,Medium,Short,Yes,No,Healthy,0,3,1
9233,Cat,2,Domestic Medium Hair,Female,White,No Color,Small,Medium,Yes,No,Healthy,0,3,1


In [13]:
test.head()

,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,PhotoAmt,target
10383,Dog,24,Mixed Breed,Female,Black,Brown,Medium,Medium,Yes,Yes,Healthy,0,1,1
10384,Dog,2,Mixed Breed,Female,Brown,No Color,Medium,Short,No,No,Healthy,0,2,0
10385,Dog,144,Mixed Breed,Male,Brown,No Color,Large,Medium,Not Sure,Not Sure,Healthy,0,1,0
10386,Dog,48,Miniature Pinscher,Female,Black,Brown,Small,Short,Yes,Yes,Healthy,0,2,1
10387,Dog,4,Mixed Breed,Male,Golden,Yellow,Medium,Short,No,No,Healthy,0,6,1


### Create an input pipeline using tf.data

Next, create a utility function that converts each training, validation and test set DataFrame into a `tf.data.Dataset`, then shuffles and batches the data.

**Note:** If you were working with a very large CSV file (so large that it does not fir into memory), you would use the `tf.data` API to read it from disk directly. That is not covered in this tutorial.

In [14]:
def df_to_dataset(dataframe, shuffle=True, batch_size=32):
    df = dataframe.copy()
    labels = df.pop('target')
    df = {key: value.to_numpy()[:, tf.newaxis] for key, value in dataframe.items()}
    ds = tf.data.Dataset.from_tensor_slices((dict(df), labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe))
    ds = ds.batch(batch_size)
    ds = ds.prefetch(batch_size)
    return ds

Now, use the newly created function (`df_to_dataset`) to check the format of the data the input pipeline helper function returns by calling it on the training data, and use a small batch size to keep the output readable:

In [15]:
batch_size = 5
train_ds = df_to_dataset(train, batch_size=batch_size)

In [16]:
[(train_features, label_batch)] = train_ds.take(1)
print("Every feature:", list(train_features.keys()))
print("A batch of ages:", train_features['Age'])
print("A batch of targets:", label_batch)

Every feature: ['Type', 'Age', 'Breed1', 'Gender', 'Color1', 'Color2', 'MaturitySize', 'FurLength', 'Vaccinated', 'Sterilized', 'Health', 'Fee', 'PhotoAmt', 'target']
A batch of ages: tf.Tensor(
[[17]
 [36]
 [36]
 [ 3]
 [ 1]], shape=(5, 1), dtype=int64)
A batch of targets: tf.Tensor([1 1 1 1 1], shape=(5,), dtype=int64)


As the output demonstrates, the training set returns a dictionary of column names (from the DataFrame) that map to column values from rows.

### Apply the Keras preprocessing layers

The Keras preprocessing layers allow you to build Keras-native input processing pipelines, which can be used as independent preprocessing code in non-Keras workflows, combined directly with Keras models, and exportes as part of a Keras SavedModel.

In this tutorial, you will use the following four preprocessing layers to demonstrate how to perform preprocessing, structured data encoding, and feature engineering:

* `tf.keras.layers.Normalization`: Performs feature-wise normalization of input features.
* `tf.keras.layers.CategoryEncoding`: Turns integer categorical features into one-hot, multi-hot, or tf-idf dense representations.
* `tf.keras.layers.StringLookup`: Turns string categorical values into integer indices.
* `tf.keras.layers.IntegerLookup`: Turns integer categorical values into integer indices.

You can learn more about the available layers in th Working with preprocessing layers guide.

* For numerical features of the PetFinder.my mini dataset, you will use a `tf.keras.layers.Normalization` layer to standardize the distribution of the data.
* For categorical features, such as pet `Type`s (`Dog` and `Cat` strings), you will transform them to multi-hot encoded tensors with `tf.keras.layers.CategoryEncoding`.

**Numerical columns**

For each numeric feature in the PetFinder.my mini dataset, you will use a `tf.keras.layers.Normalization` layer to standardize the distribution of the data.

Define a new utility function that returns a layer which applies feature-wise normalization to numerical features using that Keras preprocessing layer:

In [17]:
def get_normalization_layer(name, dataset):
    # Create a Normalization layer for the feature
    normalizer = layers.Normalization(axis=None)

    # Prepare a Dataset that only yields the feature
    feature_ds = dataset.map(lambda x, y: x[name])

    # Learn the statistics of the data
    normalizer.adapt(feature_ds)

    return normalizer

Next, test the new function by calling it on the total uploaded pet photo features to normalize `PhotoAmt`:

In [18]:
photo_count_col = train_features['PhotoAmt']
layer = get_normalization_layer('PhotoAmt', train_ds)
layer(photo_count_col)

<tf.Tensor: shape=(5, 1), dtype=float32, numpy=
array([[-0.8316896 ],
       [-0.51173705],
       [ 0.1281681 ],
       [-0.51173705],
       [-0.8316896 ]], dtype=float32)>

**Note:** If you have many numeric features (hundreds, or more), it is more efficient to concatenate them first and use a single `tf.keras.layers.Normaliation` layer.

**Categorical columns**

Pet `Type`s in the dataset are represented as strings - `Dog`s and `Cat`s - which need to be multi-hot encoded before being fed into the model.

Define another new utility function that returns a layer which maps values from a vocabulary to integer indices and multi-hot encodes the features using the `tf.keras.layers.StringLookup`, `tf.keras.layers.IntegerLookup` and `tf.keras.CategoryEncoding` preprocessing layers:

In [19]:
def get_category_encoding_layer(name, dataset, dtype, max_tokens=None):
    # Create a layer that turns strings into integer indices.
    if dtype == 'string':
        index = layers.StringLookup(max_tokens=max_tokens)
    # Otherwise, create a layer that turns integer values into integer indices.
    else:
        index = layers.IntegerLookup(max_tokens=max_tokens)

    # Prepare a 'tf.data.Dataset' that only yields the feature.
    feature_ds = dataset.map(lambda x, y: x[name])

    # Learn the set of possible values and assign them a fixed integer index.
    index.adapt(feature_ds)

    # Encode the integer indices.
    encoder = layers.CategoryEncoding(num_tokens=index.vocabulary_size())

    # Apply multi-hot encoding to the indices. The lambda function captures the layer
    # so you can use them, or include them in the Keras Functional model later.
    return lambda feature: encoder(index(feature))

Test the `get_category_encoding_layer` function by calling it on pet `Type` features to turn them into multi-hot encoded tensors:

In [20]:
test_type_col = train_features['Type']
test_type_layer = get_category_encoding_layer(name='Type', dataset=train_ds, dtype='string')
test_type_layer(test_type_col)

<tf.Tensor: shape=(5, 3), dtype=float32, numpy=
array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 1., 0.]], dtype=float32)>

Repeat the process on the pet `Age` features:

In [21]:
test_age_col = train_features['Age']
test_age_layer = get_category_encoding_layer(name='Age', dataset=train_ds, dtype='int64', max_tokens=5)
test_age_layer(test_age_col)

<tf.Tensor: shape=(5, 5), dtype=float32, numpy=
array([[1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.]], dtype=float32)>

### Preprocess selected features to train the model on

You have learned how to use several types of Keras preprocessing layers. Next, you will:

* Apply the preprocessing utility functions defined earlier on 13 numerical and categorical features from PetFinder.my mini dataset.
* Add all the feature inputs to a list.

As mentioned in the beginning, to train the model, you will use the PetFinder.my mini dataset's numerical (`PhotoAmt`, `Fee`) and categorical (`Age`, `Type`, `Color1`, `Color2`, `Gender`, `MaturitySize`, `FurLength`, `Vaccinated`, `Sterilized`, `Health`, `Breed1`) features.

**Note:** If your aim is to build an accurate model, try a larger dataset of your own, and think carefully about which features are the most meaningful to include, and how they should be represented.

Earlier, you used a small batch size to demonstrate the input pipeline. Let's now create a new input pipeline with a larger batch size of 256:

In [22]:
batch_size = 256
train_ds = df_to_dataset(train, batch_size=batch_size)
val_ds = df_to_dataset(val, shuffle=False, batch_size=batch_size)
test_ds = df_to_dataset(test, shuffle=False, batch_size=batch_size)

Normalize the numerical features (the number of pet photos and the adoption fee), and add them to one list of inputs called `encoded_features`:

In [23]:
all_inputs = {}
encoded_features = []

# Numerical features
for header in ['PhotoAmt', 'Fee']:
    numeric_col = tf.keras.Input(shape=(1,), name=header)
    normalization_layer = get_normalization_layer(header, train_ds)
    encoded_numeric_col = normalization_layer(numeric_col)
    all_inputs[header] = numeric_col
    encoded_features.append(encoded_numeric_col)

Turn the integer categorical values from the dataset (the pet age) into integer indices, perform multi-hot encoding, and add the resulting feature inputs to `encoded_features`:

In [24]:
age_col = tf.keras.Input(shape=(1,), name='Age', dtype='int64')

encoding_layer = get_category_encoding_layer(name='Age', dataset=train_ds, dtype='int64', max_tokens=5)
encoded_age_col = encoding_layer(age_col)
all_inputs['Age'] = age_col
encoded_features.append(encoded_age_col)

Repeat the same step for the string categorical values:

In [25]:
categorical_cols = ['Type', 'Color1', 'Color2', 'Gender', 'MaturitySize', 'FurLength', 'Vaccinated',
                    'Sterilized', 'Health', 'Breed1']

for header in categorical_cols:
    categorical_col = tf.keras.Input(shape=(1,), name=header, dtype='string')
    encoding_layer = get_category_encoding_layer(name=header, dataset=train_ds, dtype='string', max_tokens=5)
    encoded_categorical_col = encoding_layer(categorical_col)
    all_inputs[header] = categorical_col
    encoded_features.append(encoded_categorical_col)

### Create, compile and train the model

The next step is to create a model using the Keras Functional API. For the first layer in your model, merge the list of feature inputs - `encoded_features` - into one vector via concatenation with `tf.keras.layers.concatenate`.

In [26]:
encoded_features

[<KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor>,
 <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_1>,
 <KerasTensor shape=(None, 5), dtype=float32, sparse=False, ragged=False, name=keras_tensor_3>,
 <KerasTensor shape=(None, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_5>,
 <KerasTensor shape=(None, 5), dtype=float32, sparse=False, ragged=False, name=keras_tensor_7>,
 <KerasTensor shape=(None, 5), dtype=float32, sparse=False, ragged=False, name=keras_tensor_9>,
 <KerasTensor shape=(None, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor_11>,
 <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_13>,
 <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_15>,
 <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_17>,
 <KerasTensor shape=(None, 4), dtype=f

In [27]:
all_features = tf.keras.layers.concatenate(encoded_features)
x = tf.keras.layers.Dense(32, activation='relu')(all_features)
x = tf.keras.layers.Dropout(0.5)(x)
output = tf.keras.layers.Dense(1)(x)

model = tf.keras.Model(all_inputs, output)

Configure the model with Keras `Model.compile`:

In [28]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'],
              run_eagerly=True)

Let's visualize the connectivity graph:

In [29]:
# Use 'rankdir='LR'' to make the graph horizontal.
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, rankdir='LR')

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


Next, train and test the model:

In [30]:
model.fit(train_ds, epochs=10, validation_data=val_ds)

Epoch 1/10


C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'PhotoAmt': 'PhotoAmt', 'Fee': 'Fee', 'Age': 'Age', 'Type': 'Type', 'Color1': 'Color1', 'Color2': 'Color2', 'Gender': 'Gender', 'MaturitySize': 'MaturitySize', 'FurLength': 'FurLength', 'Vaccinated': 'Vaccinated', 'Sterilized': 'Sterilized', 'Health': 'Health', 'Breed1': 'Breed1'}
Received: inputs={'Type': 'Tensor(shape=(256, 1))', 'Age': 'Tensor(shape=(256, 1))', 'Breed1': 'Tensor(shape=(256, 1))', 'Gender': 'Tensor(shape=(256, 1))', 'Col

37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.5468 - loss: 0.6352

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'PhotoAmt': 'PhotoAmt', 'Fee': 'Fee', 'Age': 'Age', 'Type': 'Type', 'Color1': 'Color1', 'Color2': 'Color2', 'Gender': 'Gender', 'MaturitySize': 'MaturitySize', 'FurLength': 'FurLength', 'Vaccinated': 'Vaccinated', 'Sterilized': 'Sterilized', 'Health': 'Health', 'Breed1': 'Breed1'}
Received: inputs={'Type': 'Tensor(shape=(13, 1))', 'Age': 'Tensor(shape=(13, 1))', 'Breed1': 'Tensor(shape=(13, 1))', 'Gender': 'Tensor(shape=(13, 1))', 'Color1': 'Tensor(shape=(13, 1))', 'Color2': 'Tensor(shape=(13, 1))', 'MaturitySize': 'Tensor(shape=(13, 1))', 'FurLength': 'Tensor(shape=(13, 1))', 'Vaccinated': 'Tensor(shape=(13, 1))', 'Sterilized': 'Tensor(shape=(13, 1))', 'Health': 'Tensor(shape=(13, 1))', 'Fee': 'Tensor(shape=(13, 1))', 'PhotoAmt': 'Tensor(shape=(13, 1))', 'target': 'Tensor(shape=(13, 1))'}
  

37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 160ms/step - accuracy: 0.5468 - loss: 0.6352 - val_accuracy: 0.7374 - val_loss: 0.5704
Epoch 2/10


C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\models\functional.py:259: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'PhotoAmt': 'PhotoAmt', 'Fee': 'Fee', 'Age': 'Age', 'Type': 'Type', 'Color1': 'Color1', 'Color2': 'Color2', 'Gender': 'Gender', 'MaturitySize': 'MaturitySize', 'FurLength': 'FurLength', 'Vaccinated': 'Vaccinated', 'Sterilized': 'Sterilized', 'Health': 'Health', 'Breed1': 'Breed1'}
Received: inputs={'Type': 'Tensor(shape=(130, 1))', 'Age': 'Tensor(shape=(130, 1))', 'Breed1': 'Tensor(shape=(130, 1))', 'Gender': 'Tensor(shape=(130, 1))', 'Color1': 'Tensor(shape=(130, 1))', 'Color2': 'Tensor(shape=(130, 1))', 'MaturitySize': 'Tensor(shape=(130, 1))', 'FurLength': 'Tensor(shape=(130, 1))', 'Vaccinated': 'Tensor(shape=(130, 1))', 'Sterilized': 'Tensor(shape=(130, 1))', 'Health': 'Tensor(shape=(130, 1))', 'Fee': 'Tensor(shape=(130, 1))', 'PhotoAmt': 'Tensor(shape=(130, 1))', 'target': 'Tensor(shape=

37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 166ms/step - accuracy: 0.6509 - loss: 0.5915 - val_accuracy: 0.7418 - val_loss: 0.5468
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 168ms/step - accuracy: 0.6910 - loss: 0.5661 - val_accuracy: 0.7409 - val_loss: 0.5326
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 159ms/step - accuracy: 0.7022 - loss: 0.5545 - val_accuracy: 0.7426 - val_loss: 0.5240
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 155ms/step - accuracy: 0.7056 - loss: 0.5464 - val_accuracy: 0.7478 - val_loss: 0.5178
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 11s 163ms/step - accuracy: 0.7149 - loss: 0.5399 - val_accuracy: 0.7487 - val_loss: 0.5134
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - accuracy: 0.7154 - loss: 0.5375 - val_accuracy: 0.7452 - val_loss: 0.5087
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 10s 160ms/step - accuracy: 0.7195 - loss: 0.5312 - val_accuracy: 0.7418 - val_loss: 0.5058
Epoch 9/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 6s 165ms/step - accuracy: 0.7182 - loss: 0.5279 - val_accuracy: 0.7426 -

In [31]:
result = model.evaluate(test_ds, return_dict=True)
print(result)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.7244 - loss: 0.5241
{'accuracy': 0.7244367599487305, 'loss': 0.5241221189498901}


### Perform inference

The model you have developed can now classify a row from a CSV file directly after you've included the preprocessing layers inside the model itself.

You can now save and reload the Keras model with `Model.save` and `Model.load_model` before performing inference on new data:

In [32]:
model.save('my_pet_classifier.keras')

In [33]:
reloaded_model = tf.keras.models.load_model('my_pet_classifier.keras')

To get a prediction for a new sample, you can simply call the Keras `Model.predict` method. There are just two things you need to do:

1. Wrap scalars into a list so as to have a batch dimension (`Model`s only process batches of data, not single samples).

2. Call `tf.convert_to_tensor` on each feature.

In [37]:
sample = {
    'Type': 'Cat',
    'Age': 3,
    'Breed1': 'Tabby',
    'Gender': 'Male',
    'Color1': 'Black',
    'Color2': 'White',
    'MaturitySize': 'Small',
    'FurLength': 'Short',
    'Vaccinated': 'No',
    'Sterilized': 'No',
    'Health': 'Healthy',
    'Fee': 100,
    'PhotoAmt': 2,
}

In [42]:
input_dict = {name: tf.convert_to_tensor([value]) for name, value in sample.items()}
predictions = reloaded_model.predict(input_dict)
prob = tf.nn.sigmoid(predictions[0])

print("This particular pet had a %.1f percent probability of getting adopted." % (100 * prob[0]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
This particular pet had a 80.5 percent probability of getting adopted.


**Note:** You will typically have better results with deep learning with larger and more complex datasets. When working with a small dataset, such as the simplified PetFinder.my one, you can use a decision tree or a random forest as a strong baseline. The goal of this tutorial is to demonstrate the mechanics of working with structured data, so you have a starting point when working with your own datasets in the future.

### Next steps

To learn more about classifying structured data, try working with other datasets. To improve accuracy during training and testing your models, think carefully about which features to include in your model and how they should be represented.

Below are some suggestions for datasets:

* TensorFlow Datasets: MovieLens: A set of movie ratings from a movie recommendation service.
* TensorFlow Datasets: Wine Quality: Two datasets related to red and white variants of the Portuguese "Vinho Verde" wine. You can also find the Red Wine Quality dataset on Kaggle.
* Kaggle: arXiv Dataset: A corpus of 1.7 million scholarly articles from arXiv, covering physics, computer science, math, statistics, electrical engineering, quantitative biology, and economics.